# Train ResNet50 and EfficientNet-B0 on real-scan slices (Colab)

1. **Runtime > Change runtime type > T4 GPU.**
2. Put `data_real.zip` (from the Kaggle notebook `nfti-to-jpeg-conversion.ipynb`) in Google Drive under `MyDrive/brain_tumor/`.
3. Run all cells.

The two models train one after the other in the same run. Checkpoints, curves, confusion matrices and metrics are written to `MyDrive/brain_tumor/outputs/`, so they survive a Colab disconnect. If the runtime drops, just re-run all cells: a model that already finished training is skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/brain_tumor"
DATA_ZIP = f"{DRIVE_DIR}/data_real.zip"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
MODELS = ["resnet50", "efficientnet_b0"]
HEAD_EPOCHS, EPOCHS, BATCH_SIZE = 5, 10, 32

In [ ]:
# Get the code. If the repo is private, upload the repo folder to Drive instead
# and set REPO_DIR to it.
BRANCH = "main"
REPO_DIR = "/content/repo"
%cd /content
!rm -rf {REPO_DIR}
!git clone --depth 1 -b {BRANCH} https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Copy the slices to local disk (much faster to read than Drive during training)
import os
%cd {REPO_DIR}
!rm -rf /content/data_real
!unzip -q {DATA_ZIP} -d /content/data_real
!find /content/data_real -maxdepth 2 -type d

# Send outputs/ straight to Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
!rm -rf {REPO_DIR}/outputs
!ln -s {OUTPUT_DIR} {REPO_DIR}/outputs

In [ ]:
# Train each model, then evaluate it on the held-out Testing/ patients.
# The curves image is written at the very end of train.py, so it marks a finished run.
# A finished run is reused only if it was trained on the same classes as the current data.
import torch
%cd {REPO_DIR}
data_classes = sorted(os.listdir("/content/data_real/Training"))


def finished_on_same_classes(model):
    if not (os.path.exists(f"outputs/{model}_curves.png") and os.path.exists(f"outputs/{model}_best.pt")):
        return False
    ckpt_classes = torch.load(f"outputs/{model}_best.pt", map_location="cpu").get("class_names")
    if ckpt_classes != data_classes:
        print(f"{model}: saved run used classes {ckpt_classes}, data has {data_classes}; retraining")
        return False
    return True


for model in MODELS:
    if finished_on_same_classes(model):
        print(f"{model}: already trained on {data_classes}, skipping (delete outputs/{model}_curves.png to retrain)")
    else:
        !python train.py --model {model} --data_dir /content/data_real \
            --head_epochs {HEAD_EPOCHS} --epochs {EPOCHS} --batch_size {BATCH_SIZE}
    !python evaluate.py --checkpoint outputs/{model}_best.pt --data_dir /content/data_real \
        --output_csv outputs/{model}_metrics.csv

In [ ]:
# Side-by-side comparison
%cd {REPO_DIR}
csvs = " ".join(f"outputs/{m}_metrics.csv" for m in MODELS)
!python compare_models.py {csvs}

from IPython.display import Image, display
display(Image("outputs/model_comparison.png"))
for model in MODELS:
    display(Image(f"outputs/{model}_curves.png"))
    display(Image(f"outputs/{model}_confusion_matrix.png"))